In [1]:
# /// script
# requires-python = ">=3.12"
# dependencies = [
#     "matplotlib",
#     "tifffile",
#     "cellpose"
# ]
# ///

In [2]:
from pathlib import Path

import matplotlib.pyplot as plt
import tifffile
from cellpose import core, io, models, plot
from tqdm import tqdm

In [3]:
io.logger_setup()  # to get printing of progress

use_gpu = core.use_gpu()
print("GPU available:", use_gpu)

creating new log file
2026-05-28 13:53:32,065 [INFO] WRITING LOG OUTPUT TO /home/runner/.cellpose/run.log


2026-05-28 13:53:32,065 [INFO] 
cellpose version: 	4.1.1 
platform:       	linux 
python version: 	3.12.3 
torch version:  	2.12.0+cu130


2026-05-28 13:53:32,067 [INFO] Neither TORCH CUDA nor MPS version not installed/working.


GPU available: False


In [4]:
image_path = "../../../_static/images/cellpose/cell_cellpose.tif"
image = tifffile.imread(image_path)

print(image.shape)

(2, 383, 512)


In [ ]:
model = models.CellposeModel(gpu=use_gpu, model_type="cpsam")

In [ ]:
channel = 0  # channel to use for cell detection, 0 cytoplasm, 1 nucleus

# Cellpose parameters
flow_threshold = 0.4
cellprob_threshold = 0.0
tile_norm_blocksize = 0

masks, flows, styles = model.eval(
    image[channel],
    batch_size=32,
    flow_threshold=flow_threshold,
    cellprob_threshold=cellprob_threshold,
    normalize={"tile_norm_blocksize": tile_norm_blocksize},
)

In [ ]:
fig = plt.figure(figsize=(12, 5))
plot.show_segmentation(fig, image[channel], masks, flows[0])
plt.tight_layout()
# Optional if you want to also save the figure
# plt.savefig(f"path/to/output/{Path(image_path).stem}_cp_output.png")
plt.show()

In [ ]:
folder_path = Path("data/05_segmentation_cellpose")

# Read file names
images = []
for f in folder_path.glob("*.tif"):
    images.append(f)
# same as running: files = [f for f in folder_path.glob("*.tif")]

# Cellpose parameters
flow_threshold = 0.4
cellprob_threshold = 0.0
tile_norm_blocksize = 0

# Run Cellpose on all images
for image_path in tqdm(images):
    image = tifffile.imread(image_path)
    masks, flows, styles = model.eval(
        image,
        batch_size=32,
        flow_threshold=flow_threshold,
        cellprob_threshold=cellprob_threshold,
        normalize={"tile_norm_blocksize": tile_norm_blocksize},
    )

    # Optional: display each image with its segmentation
    # fig = plt.figure(figsize=(12, 5))
    # plot.show_segmentation(fig, image, masks, flows[0])
    # plt.tight_layout()
    # plt.show()

    # Save the segmentation results
    output_path = folder_path / f"{image_path.stem}_labeled_mask.tif"
    tifffile.imwrite(output_path, masks.astype("uint16"))